In [25]:
import pandas as pd
import numpy as np

import glob
import os
import os.path
import re
import sys
import warnings

from scipy.spatial import KDTree

#-------------------------------
from io import StringIO


import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap, ScalarMappable
from matplotlib.patches import Rectangle
import matplotlib.colors as colors

#from mpl_toolkits.mplot3d import Axes3D

In [26]:
#def main():

# ignore warnings
warnings.filterwarnings("ignore")

In [27]:
directory_frames = 'cascades_form_spectra_for_movie_npt'
list_files = glob.glob(f'{directory_frames}/final_pyro_cascades_*.xyz')
list_files = [list_files[0], list_files[2], list_files[6] ,list_files[11]]
list_files

['cascades_form_spectra_for_movie_npt/final_pyro_cascades_simulation200.xyz',
 'cascades_form_spectra_for_movie_npt/final_pyro_cascades_simulation350.xyz',
 'cascades_form_spectra_for_movie_npt/final_pyro_cascades_simulation500.xyz',
 'cascades_form_spectra_for_movie_npt/final_pyro_cascades_simulation0.xyz']

In [28]:
def layer_category_fix_n(zmin,zmax,n):
    layers_list = []
    beginning = zmin
    h_step = (zmax-zmin)/n
    n_layer = 0
    while (beginning < zmax):
        layers_list.append([beginning, beginning+h_step, h_step, n_layer])
        beginning = beginning + h_step 
        n_layer += 1
    return sorted(layers_list)

In [29]:
def bond_data():
    
    # ignore warnings
    warnings.filterwarnings("ignore")
    
    #layer_height = int(sys.argv[-1])
    layer_height = 2
            
    #Creatte directory to put avg results
    directory_frames = 'cascades_form_spectra_for_movie_npt'
    directory = f'{directory_frames}/results_bonds_vs_z_h{layer_height}_shrinking'
    try:
        os.makedirs(directory)
    except FileExistsError:
        # directory already exists
        pass
    
    # list of files
    list_files = glob.glob(f'{directory_frames}/final_pyro_cascades_*.xyz')
    list_files = [list_files[0], list_files[2], list_files[6] ,list_files[11]] #sim 200, 350, 500, 0
    #list_files = [list_files[0]]
    
    count_files = 1
    for filename in list_files:
        n = 36 #37
        print(count_files, "/", len(list_files), "file name:", filename[n:])
        
        border_factor = 0
        if border_factor == 0:
            file_output = f'{directory}/bonds_vs_z_h{layer_height}_shrinking_{filename[n:]}'
        #    file_output_length = f'{directory}/bonds_lenght_data_h{layer_height}_shrinking_{filename[n:]}'
        #    file_output_angl = f'{directory}/bonds_angl_data_h{layer_height}_shrinking_{filename[n:]}'
        else:
            file_output = f'{directory}/bonds_vs_z_h{layer_height}_b{border_factor}_shrinking_{filename[n:]}'
        #    file_output_length = f'{directory}/bonds_lenght_data_h{layer_height}_b{border_factor}_shrinking_{filename[n:]}'
        #    file_output_angl = f'{directory}/bonds_angl_data_h{layer_height}_b{border_factor}_shrinking_{filename[n:]}'
        
        #if (os.path.exists(file_output_length) == False) | (os.path.exists(file_output_angl) == False) :
        if os.path.exists(file_output) == False :
        #if os.path.exists(file_output) == True:
            fp = open(filename)
            for i, line in enumerate(fp):
                if i == 7:
                    z_limits = line.split()
                    z_limits = [float(i) for i in z_limits]
                elif i > 7:
                    break
            fp.close()
                    
            # load data from file
            input_data = open(filename).read()
            df_atoms = pd.read_table(filename, skiprows=9,
                                       names=['element', 'x', 'y', 'z', 'id', 'coord_num', 'pot_eng'],
                                       sep='\\s+')
            df_atoms = df_atoms.drop(['element'],axis=1)
            
            drop_list = df_atoms[(df_atoms['x'] < df_atoms['x'].min()+border_factor) | (df_atoms['x'] > df_atoms['x'].max()-border_factor) | \
                                 (df_atoms['y'] < df_atoms['y'].min()+border_factor) | (df_atoms['y'] > df_atoms['y'].max()-border_factor)   \
                                ].index
            
            df_atoms = df_atoms.drop(drop_list, axis=0).reset_index(drop=True)
            
            # assign z_layer_fix_n
            init_intf = 6
            init_z_min = -169.64307163528616
            init_z_max = 150.98293179686004
            n = (init_z_max - init_z_min) //layer_height
            
            layers = layer_category_fix_n(df_atoms['z'].min(),df_atoms['z'].max(), n)
                
            # assign z_layer group 
            def hight_category(hight):
                for i in layers:
                    cond=(i[0]< hight and hight <= i[1]);
                    if cond: 
                        return i[3]
                    else:
                        pass
            #    raise ValueError
                print("Error: Incorrect coordinate fix_n")
                print(f"{i[0]}, {i[1]}, {i[2]}, {df_atoms['z'].max()}")
            
            df_atoms['z_layers'] = df_atoms['z'].apply(hight_category)
            
            drop_list = df_atoms.loc[pd.isna(df_atoms["z_layers"]), :].index
            df_atoms = df_atoms.drop(drop_list, axis=0).reset_index(drop=True)
            
            list_layers = df_atoms['z_layers'].unique()
                        
            
            for layer_idx in list_layers:
                print( layer_idx, "/", len(list_layers), "layer")
                
                
                #----------------------------------------------------
                list_idx_central_atom = []
                list_x_central_atom = []
                list_y_central_atom = []
                list_z_central_atom = []
                list_lmp_id_central_atom = []
                list_z_layers_central_atom = []                
                
                list_idx_neigb1_atom = []
                list_x_neigb1_atom = []
                list_y_neigb1_atom = []
                list_z_neigb1_atom = []
                list_lmp_id_neigb1_atom = []
                
                #----------------------------------------------------
                list_angl_idx_central_atom = []
                list_angl_x_central_atom = []
                list_angl_y_central_atom = []
                list_angl_z_central_atom = []
                list_angl_lmp_id_central_atom = []
                list_angl_z_layers_central_atom = []                
                
                list_angl_idx_neigb1_atom = []
                list_angl_x_neigb1_atom = []
                list_angl_y_neigb1_atom = []
                list_angl_z_neigb1_atom = []
                list_angl_lmp_id_neigb1_atom = []
                
                list_angl_idx_neigb2_atom = []
                list_angl_x_neigb2_atom = []
                list_angl_y_neigb2_atom = []
                list_angl_z_neigb2_atom = []
                list_angl_lmp_id_neigb2_atom = []
                #----------------------------------------------------
                
                
                
                if border_factor == 0:
                    #file_output = f'{directory}/bonds_vs_z_h{layer_height}_shrinking_{filename[n:]}'
                    n = 36
                    file_output_length = f'{directory}/bonds_lenght_data_h{layer_height}_layer{layer_idx}_shrinking_{filename[n:]}'
                    file_output_angl = f'{directory}/bonds_angl_data_h{layer_height}_layer{layer_idx}_shrinking_{filename[n:]}'
                else:
                    #file_output = f'{directory}/bonds_vs_z_h{layer_height}_b{border_factor}_shrinking_{filename[n:]}'
                    file_output_length = f'{directory}/bonds_lenght_data_h{layer_height}_layer{layer_idx}_b{border_factor}_shrinking_{filename[n:]}'
                    file_output_angl = f'{directory}/bonds_angl_data_h{layer_height}_layer{layer_idx}_b{border_factor}_shrinking_{filename[n:]}'
                
                layer_min = layer_idx - 1
                layer_max = layer_idx + 1
                if layer_idx == min(list_layers): 
                    layer_min = max(list_layers)
                    df_atoms.loc[(df_atoms['z_layers']==layer_min),'z'] = df_atoms['z'].subtract((df_atoms[df_atoms['z_layers']==layer_idx]['z'].max() - z_limits[0]))
                    
                if layer_idx == max(list_layers): 
                    layer_max = min(list_layers)
                    df_atoms.loc[(df_atoms['z_layers']==layer_max),'z'] = df_atoms['z'].add((z_limits[1] - df_atoms[df_atoms['z_layers']==layer_idx]['z'].min()))
                    
                
                df_slice = df_atoms[df_atoms['z_layers'].isin([layer_min, layer_idx, layer_max])==True].copy()
                df_slice = df_slice.reset_index(drop=True)
                
                # ---------find all neighbours for all atoms in a sphere of 2 angstrom --------                
                # KDTree                
                kd_tree = KDTree(df_slice[['x','y','z']].to_numpy())
                
                atoms_idx = list(df_slice[df_slice['z_layers']==layer_idx].index)
                for idx in atoms_idx:
                    ii = kd_tree.query_ball_point(df_slice.loc[idx][['x','y','z']].to_numpy(), r=2)
                    if idx in atoms_idx:
                        ii.remove(idx)
                        ii_copy = ii.copy()
                        if len(ii) > 0:
                            #for i in ii:
                            #for i, elem in enumerate(ii):
                            for elem in ii:
                                list_idx_central_atom.append(idx)
                                list_x_central_atom.append(df_slice.loc[idx,'x'])
                                list_y_central_atom.append(df_slice.loc[idx,'y'])
                                list_z_central_atom.append(df_slice.loc[idx,'z'])
                                list_lmp_id_central_atom.append(df_slice.loc[idx,'id'])
                                list_z_layers_central_atom.append(df_slice.loc[idx,'z_layers'])
                                
                                list_idx_neigb1_atom.append(elem)
                                list_x_neigb1_atom.append(df_slice.loc[elem,'x'])
                                list_y_neigb1_atom.append(df_slice.loc[elem,'y'])
                                list_z_neigb1_atom.append(df_slice.loc[elem,'z'])
                                list_lmp_id_neigb1_atom.append(df_slice.loc[elem,'id'])
                                
                                if len(ii) > 1:
                                    #nextelem = ii[(i + 1) % len(ii)]
                                    
                                    ii_copy.remove(elem)
                                    for kk in ii_copy:                                        
                                        list_angl_idx_central_atom.append(idx)
                                        list_angl_x_central_atom.append(df_slice.loc[idx,'x'])
                                        list_angl_y_central_atom.append(df_slice.loc[idx,'y'])
                                        list_angl_z_central_atom.append(df_slice.loc[idx,'z'])
                                        list_angl_lmp_id_central_atom.append(df_slice.loc[idx,'id'])
                                        list_angl_z_layers_central_atom.append(df_slice.loc[idx,'z_layers'])
                                        
                                        list_angl_idx_neigb1_atom.append(elem)
                                        list_angl_x_neigb1_atom.append(df_slice.loc[elem,'x'])
                                        list_angl_y_neigb1_atom.append(df_slice.loc[elem,'y'])
                                        list_angl_z_neigb1_atom.append(df_slice.loc[elem,'z'])
                                        list_angl_lmp_id_neigb1_atom.append(df_slice.loc[elem,'id'])
                                        
                                        list_angl_idx_neigb2_atom.append(kk)
                                        list_angl_x_neigb2_atom.append(df_slice.loc[kk,'x'])
                                        list_angl_y_neigb2_atom.append(df_slice.loc[kk,'y'])
                                        list_angl_z_neigb2_atom.append(df_slice.loc[kk,'z'])
                                        list_angl_lmp_id_neigb2_atom.append(df_slice.loc[kk,'id'])
                
                data_bonds_length = {'idx_central_atom': list_idx_central_atom,
                          'x_central_atom': list_x_central_atom,
                          'y_central_atom': list_y_central_atom,
                          'z_central_atom': list_z_central_atom,
                          'lmp_id_central_atom': list_lmp_id_central_atom,
                          'z_layers_central_atom': list_z_layers_central_atom,
                          'idx_neigb1_atom': list_idx_neigb1_atom,
                          'x_neigb1_atom': list_x_neigb1_atom,
                          'y_neigb1_atom': list_y_neigb1_atom,
                          'z_neigb1_atom': list_z_neigb1_atom,
                          'lmp_id_neigb1_atom': list_lmp_id_neigb1_atom
                       }
            
                data_bonds_length = pd.DataFrame(data_bonds_length)
            
                data_bonds_angl = {'idx_central_atom': list_angl_idx_central_atom,
                          'x_central_atom': list_angl_x_central_atom,
                          'y_central_atom': list_angl_y_central_atom,
                          'z_central_atom': list_angl_z_central_atom,
                          'lmp_id_central_atom': list_angl_lmp_id_central_atom,
                          'z_layers_central_atom': list_angl_z_layers_central_atom,
                          'idx_neigb1_atom': list_angl_idx_neigb1_atom,
                          'x_neigb1_atom': list_angl_x_neigb1_atom,
                          'y_neigb1_atom': list_angl_y_neigb1_atom,
                          'z_neigb1_atom': list_angl_z_neigb1_atom,
                          'lmp_id_neigb1_atom': list_angl_lmp_id_neigb1_atom,
                          'idx_neigb2_atom': list_angl_idx_neigb2_atom,
                          'x_neigb2_atom': list_angl_x_neigb2_atom,
                          'y_neigb2_atom': list_angl_y_neigb2_atom,
                          'z_neigb2_atom': list_angl_z_neigb2_atom,
                          'lmp_id_neigb2_atom': list_angl_lmp_id_neigb2_atom
                       }
            
                data_bonds_angl = pd.DataFrame(data_bonds_angl)
                #display(data_bonds_angl[['idx_central_atom','idx_neigb1_atom','idx_neigb2_atom']])
            
                # write results into file
                data_bonds_angl.to_csv(file_output_angl)
                data_bonds_length.to_csv(file_output_length)
                # ----------calculate bond length for each pair, its total mean and dispersion---------
                
                # ----------calculate angle for each triplet, its total mean and dispersion-----------
            
            ##count atoms in each layer
            #df_atoms_grouped = df_atoms.groupby('z_layers')['id'].count()
            #df_atoms_grouped = df_atoms_grouped.reset_index()
            #df_atoms_grouped = df_atoms_grouped.rename(columns={"id": "atoms_total"})
            #
            ##count defect in each layer
            #df_poteng = df_atoms[df_atoms['pot_eng'] > -7.0].groupby('z_layers')['pot_eng'].count()
            #df_poteng = df_poteng.reset_index()
            #df_poteng = df_poteng.rename(columns={"pot_eng": "def_pot_eng"})
            #
            #df_coordnum = df_atoms[df_atoms['coord_num'] != 3].groupby('z_layers')['coord_num'].count()
            #df_coordnum = df_coordnum.reset_index()
            #df_coordnum = df_coordnum.rename(columns={"coord_num": "def_coord_num"})
            #
            #df_coordnum_1 = df_atoms[df_atoms['coord_num'] == 1].groupby('z_layers')['coord_num'].count()
            #df_coordnum_1 = df_coordnum_1.reset_index()
            #df_coordnum_1 = df_coordnum_1.rename(columns={"coord_num": "coord_num_1"})
            #
            #
            ## merge result in one dataframe
            #array=np.array(layers)        
            #df_def = pd.DataFrame(array,columns=['beginning','ending','height','z_layers'])
            #df_def = df_def.merge(df_atoms_grouped, how='outer')
            #df_def = df_def.merge(df_coordnum, how='outer')
            #df_def = df_def.merge(df_poteng,how='outer')
            #df_def = df_def.merge(df_coordnum_1, how='outer')
            #df_def = df_def.fillna(0)
            #
            ## write results into file
            #df_def.to_csv(file_output)
        
        count_files += 1
        print("Done")

In [30]:
bond_data()

1 / 4 file name: final_pyro_cascades_simulation200.xyz
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 196.5540000000005, 2.57355, 196.554
Error: Incorrect coordinate fix_n
193.9804500000005, 19

153.0 / 161 layer
154.0 / 161 layer
155.0 / 161 layer
156.0 / 161 layer
157.0 / 161 layer
158.0 / 161 layer
159.0 / 161 layer
160.0 / 161 layer
Done
3 / 4 file name: final_pyro_cascades_simulation500.xyz
Error: Incorrect coordinate fix_n
196.10356875000082, 198.70400000000083, 2.60043125, 198.704
Error: Incorrect coordinate fix_n
196.10356875000082, 198.70400000000083, 2.60043125, 198.704
0.0 / 160 layer
1.0 / 160 layer
2.0 / 160 layer
3.0 / 160 layer
4.0 / 160 layer
5.0 / 160 layer
6.0 / 160 layer
7.0 / 160 layer
8.0 / 160 layer
9.0 / 160 layer
10.0 / 160 layer
11.0 / 160 layer
12.0 / 160 layer
13.0 / 160 layer
14.0 / 160 layer
15.0 / 160 layer
16.0 / 160 layer
17.0 / 160 layer
18.0 / 160 layer
19.0 / 160 layer
20.0 / 160 layer
21.0 / 160 layer
22.0 / 160 layer
23.0 / 160 layer
24.0 / 160 layer
25.0 / 160 layer
26.0 / 160 layer
27.0 / 160 layer
28.0 / 160 layer
29.0 / 160 layer
30.0 / 160 layer
31.0 / 160 layer
32.0 / 160 layer
33.0 / 160 layer
34.0 / 160 layer
35.0 / 160 layer
36.0 /